# QCC | Reproducible Circuit Portfolio - expected GM 27.312

This notebook assembles a submission from **public, credited circuits**, checks every member against its pinned
SHA-256, writes `/kaggle/working/submission.zip`, and re-runs the **unmodified official verifiers** for chemistry
(`dfthcsos`) and `invsqrt` with default shots and salt. Members are located in the attached datasets by file name and
accepted only if their SHA-256 matches the pin, so a change in an upstream dataset cannot silently change the result.

**Changes vs the previous version (GM 27.28999):** the invsqrt member and the chemistry member are replaced by two
cheaper public files (below); the DQI member is unchanged.

| Task | Member | Cost | Reference / cost |
|---|---|---:|---:|
| invsqrt | `invsqrt.kmb` `1ba197b0` | 78,673.57 (re-verified here) | 26.397x |
| DQI | `dqi.kmb` `821bf8d1` (cutoff 18) | 360,209,905,486.02 (zero-error verifier receipt, not re-run: needs many GB and hours) | 67.848x |
| dfthcsos | 15 loaders (4,033 Toffoli per pass) + `submission.npz` `54bc3e71` | 294,984,216.56 (re-verified here) | 11.376x |

Expected geometric mean 27.312 if DQI draws zero failures (one failed DQI shot costs about 0.15% of the score).

## Credits
- **invsqrt:** `1ba197b02771` is entry HW-26 of the dataset
  [manwithacat/vqcc-hedge-wizardry](https://www.kaggle.com/datasets/manwithacat/vqcc-hedge-wizardry) (James Barlow / manwithacat):
  the measure-first edit by Asterios Panoras (asteriospanoras,
  [QCC 27.30 | What Works, What Fails and Why](https://www.kaggle.com/code/asteriospanoras)) on the 46-qubit line:
  full-domain angle refits by Ertugrul Ozer and Atakan Aldemir of the 64-row-table build by Ali Haydar Ozdag (ahoirg) of
  Syed Asad Ali's generator (Apache-2.0).
- **Chemistry:** loaders `ede3b67ad7ff` by maoxin123,
  [QCC v27 | 4033-Toffoli coded-word chemistry](https://www.kaggle.com/code/maoxin123/qcc-v27-4033-toffoli-coded-word-chemistry);
  weights `c6e74b63ad82` by Asterios Panoras,
  [QCC 27.307 frontier: finer chemistry refit](https://www.kaggle.com/code/asteriospanoras/qcc-27-307-frontier-finer-chemistry-refit);
  both taken from the dataset
  [manwithacat/vqcc-apache-public-circuits](https://www.kaggle.com/datasets/manwithacat/vqcc-apache-public-circuits), whose
  assembly record (`parts/assembly_best.json`) this notebook reproduces. Earlier lineage credited by those authors:
  Heliosli, prvsiyan, Leo Tang, Ertugrul Ozer, Syed Asad Ali, Lera/marculera, Ahmed Berat Ozer, Atakan Aldemir, busyaprime.
- **DQI:** Syed Asad Ali's (syed3000) v12 decoder, [VQCC Own-Circuit Portfolio](https://www.kaggle.com/code/syed3000/vqcc-own-circuit-portfolio-notebook) (Apache-2.0);
  extractor cutoff idea by Ertugrul Ozer (ozertuu), [Quantum AI 27.083+ VQCC: DQI cutoff 17](https://www.kaggle.com/code/ozertuu/quantum-ai-27-083-vqcc-dqi-cutoff-17);
  cutoff 18 with 241 retained products by Areeb Ali (areebbb),
  [QAI fewer Toffolis, same qubits](https://www.kaggle.com/code/areebbb/qai-fewer-toffolis-same-qubits-lb-27-11042).
  File as shipped in [ahoirg/qcc-portfolio-gm-27-28999](https://www.kaggle.com/datasets/ahoirg/qcc-portfolio-gm-27-28999).
- **Specifications, verifiers, kickmix:** Google Quantum AI.


In [1]:
import hashlib, json, os, pathlib, shutil, subprocess, sys, time, zipfile

EXPECTED = {
 "dfthcsos/angle_k_0.kmb": "71735b23037e8a64d16e39b3347d440b36bdf8813613ec70c36a85bfd362d7c2",
 "dfthcsos/angle_k_1.kmb": "e63f9de0a60ff79fc6f817056a1f95f5efed9ae82955cf56b11492e5ff28d70b",
 "dfthcsos/angle_k_10.kmb": "c2e7dca4c51cdb3f82a47effbf0d529351b78a432e0b30bf5ca833bf2c5fcea3",
 "dfthcsos/angle_k_11.kmb": "18fa8cd8125a1eccb959be27956ce7e871599909193052ab4a2ad1a5d0d6b01d",
 "dfthcsos/angle_k_12.kmb": "62d6cd9855bcad70917a56527c0f03b912fba8bb14a2bfd91748a345f0a6fc72",
 "dfthcsos/angle_k_13.kmb": "09a2faee4582e786548640ce4552bcc812e73667bd6784b37e0bbb5b817b8189",
 "dfthcsos/angle_k_14.kmb": "a582eb78249c7ba5f58881b2753e2900a5a5be61eda719d2a24aff1fdad5f073",
 "dfthcsos/angle_k_2.kmb": "e32afaf5d4b04756d20f15ecfbf08d6db325b57cbcb877428cb7fd61f13fcebb",
 "dfthcsos/angle_k_3.kmb": "a932bc25467908c2189aadadd5f997ea88200b44d9401741c2b492fc1c2ebaa0",
 "dfthcsos/angle_k_4.kmb": "dc70bda726f221813b502fc2d26acdd21882d1e30177f4264974880eb68cb2c2",
 "dfthcsos/angle_k_5.kmb": "dbef059b4cd3030d2b249df4d09653efb54f8b803faa2d459f4e418394120fe2",
 "dfthcsos/angle_k_6.kmb": "11cb19cb7e9d628f1416ef0e03f6d9d767726e42df839885e9224b029444f383",
 "dfthcsos/angle_k_7.kmb": "e3a11ed9c462603f6f0734fe61fc0e4e924fa6bb4a6702574391f28f507a264b",
 "dfthcsos/angle_k_8.kmb": "0fdcfe1906582661c40d0cd510df7efa2c115f287f9ad859223967d98150b0b2",
 "dfthcsos/angle_k_9.kmb": "e0e95ba89402beb2c08017c5db7bc5070db26fd996bcd4f465d2ab1a21cfa55f",
 "dfthcsos/submission.npz": "54bc3e71807e9a21338db19fe5a5cfc1b933338ec7f323dbee3ce283ffccbfa2",
 "dqi.kmb": "821bf8d157c03c3e868a0099b0ab1438cfba664ab1bc6fec3210654099b672a3",
 "invsqrt.kmb": "1ba197b027713aac164056d12505c87f286c3fcd94f403c522073325763bf2f9"
}

INPUT = pathlib.Path("/kaggle/input"); OUT = pathlib.Path("/kaggle/working")
T0 = time.time()

def sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 22), b""):
            h.update(chunk)
    return h.hexdigest()

# Each member is searched by file name in every attached dataset and accepted only if its SHA-256 equals the pin.
NAMES = {"invsqrt.kmb": "1ba197b02771.kmb", "dfthcsos/submission.npz": "c6e74b63ad82.npz", "dqi.kmb": "dqi.kmb"}
NAMES.update({f"dfthcsos/angle_k_{i}.kmb": f"angle_k_{i}.kmb" for i in range(15)})
by_name = {}
for p in INPUT.rglob("*"):
    if p.is_file() and p.suffix in (".kmb", ".npz"):
        for member, name in NAMES.items():
            if p.name == name or p.name.endswith("__" + name):
                by_name.setdefault(member, []).append(p)
src = {}
for member in EXPECTED:
    hit = next((p for p in sorted(by_name.get(member, [])) if sha(p) == EXPECTED[member]), None)
    assert hit is not None, f"{member}: none of {len(by_name.get(member, []))} candidate files matches its pinned SHA-256"
    src[member] = hit
    print(f"{member:26s} <- {hit.relative_to(INPUT)}")

dst = OUT / "submission.zip"
with zipfile.ZipFile(dst, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zo:
    for n in sorted(EXPECTED):
        zo.write(src[n], n)
with zipfile.ZipFile(dst) as z:
    assert z.testzip() is None
    assert sorted(z.namelist()) == sorted(EXPECTED)
    for n in z.namelist():
        assert hashlib.sha256(z.read(n)).hexdigest() == EXPECTED[n]
print("wrote", dst, round(dst.stat().st_size / 1e6, 2), "MB; all", len(EXPECTED), "members match their pins")


dfthcsos/angle_k_0.kmb     <- vqcc-apache-public-circuits/circuits/chemistry/657475541005/angle_k_0.kmb
dfthcsos/angle_k_1.kmb     <- vqcc-apache-public-circuits/circuits/chemistry/657475541005/angle_k_1.kmb
dfthcsos/angle_k_10.kmb    <- vqcc-apache-public-circuits/circuits/chemistry/657475541005/angle_k_10.kmb
dfthcsos/angle_k_11.kmb    <- vqcc-apache-public-circuits/circuits/chemistry/657475541005/angle_k_11.kmb
dfthcsos/angle_k_12.kmb    <- vqcc-apache-public-circuits/circuits/chemistry/657475541005/angle_k_12.kmb
dfthcsos/angle_k_13.kmb    <- vqcc-apache-public-circuits/circuits/chemistry/657475541005/angle_k_13.kmb
dfthcsos/angle_k_14.kmb    <- vqcc-apache-public-circuits/circuits/chemistry/657475541005/angle_k_14.kmb
dfthcsos/angle_k_2.kmb     <- vqcc-apache-public-circuits/circuits/chemistry/657475541005/angle_k_2.kmb
dfthcsos/angle_k_3.kmb     <- vqcc-apache-public-circuits/circuits/chemistry/657475541005/angle_k_3.kmb
dfthcsos/angle_k_4.kmb     <- vqcc-apache-public-circuits/c

In [2]:
# unmodified official verifiers, default shots and salt
engine = next(p.parent for p in INPUT.rglob("kickmix.so") if (p.parent.parent / "verifiers").is_dir())
verifiers = engine.parent / "verifiers"
py = next(c for c in [shutil.which("python3.13"), sys.executable] if c)
env = dict(os.environ, PYTHONPATH=f"{engine}:{verifiers}")
run = pathlib.Path("/tmp/vqcc"); shutil.rmtree(run, ignore_errors=True); run.mkdir()
with zipfile.ZipFile(dst) as z:
    z.extractall(run)

def verify(cmd, name):
    t = time.time()
    r = subprocess.run([py] + cmd, capture_output=True, text=True, env=env, cwd=str(verifiers))
    print(r.stderr[-1500:])
    res = json.loads(r.stdout[r.stdout.index("{"):])
    res["wall_seconds"] = round(time.time() - t, 1)
    (OUT / f"{name}_verify.json").write_text(json.dumps(res, indent=1))
    keys = ("status", "score", "lambda_eff", "toffoli_count", "t_count", "num_qubits", "verified_shots", "wall_seconds")
    print(name, json.dumps({k: res.get(k) for k in keys}))
    assert res["status"] == "PASS", f"{name} verifier did not pass"
    return res

chem = verify([str(verifiers / "dfthcsos" / "verify.py"), str(run / "dfthcsos")], "dfthcsos")
inv = verify([str(verifiers / "invsqrt" / "verify_submission_invsqrt.py"), str(run / "invsqrt.kmb"),
              "--max-workers", str(os.cpu_count())], "invsqrt")
# the costs this version was assembled for (official verifier, local run of 2026-11-14)
assert abs(float(chem["score"]) - 294984216.5610552) < 1e-3 * 294984216.56 * 1e-6, chem["score"]
assert abs(float(inv["score"]) - 78673.56921101942) < 1e-3 * 78673.57 * 1e-6 + 1e-6, inv["score"]



dfthcsos {"status": "PASS", "score": 294984216.5610552, "lambda_eff": 46.905862777117996, "toffoli_count": 14728.0, "t_count": null, "num_qubits": 427, "verified_shots": 78080, "wall_seconds": 27.1}

invsqrt {"status": "PASS", "score": 78673.56921101942, "lambda_eff": null, "toffoli_count": 1478.750911494735, "t_count": 1.0, "num_qubits": 46, "verified_shots": 200540928, "wall_seconds": 580.5}


In [3]:
import importlib.metadata, resource
REF = {"invsqrt": 2076774.233715282, "dqi": 24439463583672.93, "dfthcsos": 3355599588.3155375}
COST = {"invsqrt": float(inv["score"]), "dqi": 360209905486.0156, "dfthcsos": float(chem["score"])}
gm = 1.0
for k in REF:
    gm *= REF[k] / COST[k]
    print(f"{k:9s} cost {COST[k]:>22,.2f}  ratio {REF[k] / COST[k]:.5f}")
print(f"expected geometric mean {gm ** (1 / 3):.5f} (DQI cost from the zero-error receipt, not re-run here)")
sys.path.insert(0, str(engine))
try:
    import kickmix; kv = getattr(kickmix, "__version__", "unknown")
except Exception as e:
    kv = f"import failed: {e}"
print("kickmix:", kv)
print("verifier sha256:", {p.name: sha(p)[:16] for p in [verifiers / "dfthcsos" / "verify.py", verifiers / "invsqrt" / "verify_submission_invsqrt.py"]})
print("member sha256:", {n: h[:8] for n, h in EXPECTED.items() if not n.startswith("dfthcsos/angle")})
print("salt: default (empty); shots: verifier defaults")
print("wall seconds:", round(time.time() - T0, 1), "| peak RSS MB:", round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024, 1),
      "| peak child RSS MB:", round(resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss / 1024, 1))


invsqrt   cost              78,673.57  ratio 26.39736
dqi       cost     360,209,905,486.02  ratio 67.84784
dfthcsos  cost         294,984,216.56  ratio 11.37552
expected geometric mean 27.31215 (DQI cost from the zero-error receipt, not re-run here)
kickmix: 0.1
verifier sha256: {'verify.py': '0775be7a52892236', 'verify_submission_invsqrt.py': 'ddef520dec6b68d1'}
member sha256: {'dfthcsos/submission.npz': '54bc3e71', 'dqi.kmb': '821bf8d1', 'invsqrt.kmb': '1ba197b0'}
salt: default (empty); shots: verifier defaults
wall seconds: 775.9 | peak RSS MB: 1021.6 | peak child RSS MB: 1021.6
